# OCR thô hai luồng Hán/Việt cho một work, một khoảng trang (Colab hoặc máy local)

Notebook này làm toàn bộ quy trình trong `docs/ocr_runbook.md` cho **một người**:
PDF → ảnh trang → paddlevl (Hán) → Qwen (Việt) → `build_layers` → `check_ocr_complete` → gói `.zip` gửi về.

**Mỗi người chỉ sửa ô 1** (`WORK`, `PAGES`, `MEMBER`, và đường dẫn tuỳ nơi chạy). Sau đó *Run all*.

| Người | `WORK` | `PAGES` | `MEMBER` |
|---|---|---|---|
| Thành viên 1 | `CPO_18` | `205-544` | `tv1` |
| Thành viên 2 | `CPO_19` | `5-623` | `tv2` |
| Thành viên 3 | `CPO_19` | `624-833` | `tv3` |
| Thành viên 4 | `CPO_20` | `5-806` | `tv4` |

### Chạy trên Colab
Runtime **GPU (T4)** (*Runtime → Change runtime type → T4 GPU*). Drive còn trống ~2 GB (thêm ~9 GB nếu lưu model
trên Drive). Notebook tự cài hai môi trường, tải model, lưu kết quả lên Drive.

### Chạy trên máy local (Windows/Linux, GPU NVIDIA)
Đã cài theo `docs/ocr_runbook.md` mục 3 (`.venv-paddle` + conda `my_nlp`, model đã tải). Mở notebook từ repo bằng
Jupyter / VS Code, **kernel `my_nlp`** (chưa có: `pip install ipykernel notebook` trong `my_nlp`). Notebook bỏ qua
cài đặt/Drive, dùng môi trường có sẵn, gọi `scripts/ocr_work.ps1` (Windows) hoặc `scripts/ocr_work.sh` (Linux).
Kết quả ở `data/<WORK>/03_ocr`, gói ở `dist/`. Đừng chạy khi một lệnh OCR khác đang dùng GPU.

**Bị ngắt (Colab ngắt phiên, tắt máy, OOM)?** *Run all* lại: mọi bước resume (`--skip-done`), trang đã xong không
bị OCR lại. Các ô chạy tuần tự nên hai model không bao giờ chạy cùng lúc.

## 1. Tham số — mỗi người CHỈ sửa ô này

In [ ]:
WORK   = "CPO_19"          # CPO_18 | CPO_19 | CPO_20
PAGES  = "5-623"           # khoảng trang PDF (1-based), đúng như bảng phân công
MEMBER = "tv2"             # mã người chạy, có trong tên gói gửi về
RUN_MODE = "auto"          # auto (tự nhận Colab) | colab | local

# ---- chỉ dùng khi chạy LOCAL (để trống = tự tìm)
PADDLE_PY = ""             # python của .venv-paddle, vd r"E:\Projects\han-viet-corpus\.venv-paddle\Scripts\python.exe"
QWEN_PY   = ""             # python của my_nlp; trống = kernel hiện tại nếu là my_nlp, rồi `conda env list`
LOCAL_DATA_ROOT = "data"   # thư mục data (tương đối với gốc repo)

# ---- chỉ dùng khi chạy trên COLAB
PDF_ON_DRIVE = "/content/drive/MyDrive/han_viet_ocr/NgoThiNhamToanTap3_LamGiang.pdf"   # PDF nguồn (đúng bản nhóm phát)
DRIVE_ROOT = "/content/drive/MyDrive/han_viet_ocr"   # lưu 03_ocr, log, gói zip: không mất khi phiên bị ngắt
REPO_URL = "https://github.com/khangthinhne/chinese-vietnamese-corpus.git"
GIT_REF  = "ocr-v1"        # tag/branch/commit chung do người điều phối thông báo
REPO_ZIP = ""              # hoặc zip của repo trên Drive (bỏ trống = dùng git)
# Repo private: thêm secret GITHUB_TOKEN trong Colab (biểu tượng chìa khoá), KHÔNG dán token vào đây.
HF_CACHE_ON_DRIVE = False  # True: lưu model (~9 GB) trên Drive
KEEP_IMAGES_ON_DRIVE = False

# ---- cả hai
SEED_ZIP = ""              # gói hạt giống (trang đã OCR sẵn), vd ".../CPO_19_0015-0121_seed_20261125.zip"

## 2. Nơi chạy, thư mục, hàm chạy lệnh, kiểm tra GPU

In [ ]:
import os, re, json, shutil, subprocess, sys
from pathlib import Path

assert re.fullmatch(r"CPO_\d+", WORK), "WORK dạng CPO_xx"
assert re.fullmatch(r"\d+(-\d+)?(,\d+(-\d+)?)*", PAGES), "PAGES dạng 5-623"
try:
    import google.colab  # noqa: F401
    _in_colab = True
except ImportError:
    _in_colab = False
IS_COLAB = _in_colab if RUN_MODE == "auto" else RUN_MODE == "colab"
IS_WINDOWS = os.name == "nt"
print("Chạy trên:", "Colab" if IS_COLAB else ("local Windows" if IS_WINDOWS else "local Linux/macOS"))

if IS_COLAB:
    REPO = Path("/content/han-viet-corpus")
    DRIVE = Path(DRIVE_ROOT)
    HF_CACHE = (DRIVE / "hf_cache") if HF_CACHE_ON_DRIVE else Path("/content/hf_cache")
    DATA_ROOT = "data"
    DIST = DRIVE / "dist"
    PADDLE_PY = "/content/venv-paddle/bin/python"
    QWEN_PY = "/content/venv-qwen/bin/python"
else:
    here = Path.cwd().resolve()
    REPO = next((p for p in (here, *here.parents) if (p / "configs").is_dir() and (p / "src" / "ocr").is_dir()), None)
    assert REPO, "Mở notebook từ trong repo (thư mục có configs/ và src/)"
    HF_CACHE = None                                      # giữ cache HF sẵn có của máy
    DATA_ROOT = LOCAL_DATA_ROOT
    DIST = REPO / "dist"
    if not PADDLE_PY:
        PADDLE_PY = str(REPO / ".venv-paddle" / ("Scripts/python.exe" if IS_WINDOWS else "bin/python"))
    if not QWEN_PY:
        prefix = Path(sys.prefix)
        if prefix.name == "my_nlp":
            QWEN_PY = sys.executable
        else:
            try:
                listing = subprocess.run(["conda", "env", "list"], capture_output=True, text=True, shell=IS_WINDOWS).stdout
            except OSError:
                listing = ""
            m = re.search(r"^\s*my_nlp\s+(?:[*+]\s+)?(.+?)\s*$", listing, re.M)
            if m:
                QWEN_PY = str(Path(m[1]) / ("python.exe" if IS_WINDOWS else "bin/python"))
    for name, py in (("PADDLE_PY", PADDLE_PY), ("QWEN_PY", QWEN_PY)):
        assert py and Path(py).exists(), f"Không thấy {name} ({py}); đặt đường dẫn ở ô 1 (docs/ocr_runbook.md mục 3)"
    poppler = Path(QWEN_PY).parent / "Library" / "bin"
    if poppler.is_dir():
        os.environ["PATH"] = f"{poppler}{os.pathsep}{os.environ['PATH']}"
print("REPO      ", REPO)
print("PADDLE_PY ", PADDLE_PY)
print("QWEN_PY   ", QWEN_PY)

def run(cmd, cwd=None, env=None, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> dừng notebook (trừ khi check=False)."""
    print(">>", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    full_env = {**os.environ, "PYTHONUTF8": "1", **(env or {})}
    if HF_CACHE is not None:
        full_env["HF_HUB_CACHE"] = str(HF_CACHE)
    p = subprocess.Popen(cmd, cwd=cwd, env=full_env, shell=isinstance(cmd, str), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if check and code != 0:
        raise SystemExit(f"Lệnh lỗi (mã {code}). Xem output ở trên; bảng xử lý lỗi: docs/ocr_runbook.md mục 5")
    return code

def ocr_step(steps, dry_run=False, pack=False, check=True):
    """Một hoặc vài bước của scripts/ocr_work (ps1 trên Windows, sh nơi khác) cho WORK/PAGES."""
    steps = steps.split()
    if IS_WINDOWS and not IS_COLAB:
        cmd = ["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-File", str(REPO / "scripts" / "ocr_work.ps1"),
               "-Work", WORK, "-Pages", PAGES, "-Steps", ",".join(steps), "-DataRoot", DATA_ROOT,
               "-PaddlePython", str(PADDLE_PY), "-QwenPython", str(QWEN_PY)]
        if dry_run:
            cmd.append("-DryRun")
        if pack:
            cmd += ["-Pack", "-Member", MEMBER]
        return run(cmd, cwd=REPO, check=check)
    env = {"PADDLE_PY": str(PADDLE_PY), "QWEN_PY": str(QWEN_PY), "DATA_ROOT": DATA_ROOT, "STEPS": " ".join(steps)}
    if dry_run:
        env["DRY_RUN"] = "1"
    if pack:
        env.update(PACK="1", MEMBER=MEMBER)
    return run(["bash", "scripts/ocr_work.sh", WORK, PAGES], cwd=REPO, env=env, check=check)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("KHÔNG có GPU NVIDIA. Colab: Runtime -> Change runtime type -> T4 GPU. Local: kiểm tra driver.")
print(gpu.stdout)

## 3. (Colab) Drive + mã nguồn đúng tag/commit chung của nhóm

In [ ]:
if not IS_COLAB:
    print("Local: bỏ qua (dùng repo đang mở). Kiểm tra commit:")
    run(["git", "log", "-1", "--format=commit %H (%d) %s"], cwd=REPO, check=False)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    for d in (DRIVE / "logs", DRIVE / "dist"):
        d.mkdir(parents=True, exist_ok=True)
    if REPO_ZIP:
        if not REPO.exists():
            tmp = Path("/content/repo_unzip"); shutil.rmtree(tmp, ignore_errors=True)
            shutil.unpack_archive(REPO_ZIP, tmp)
            top = [p for p in tmp.iterdir() if p.is_dir()]
            shutil.move(str(top[0] if len(top) == 1 and not (tmp / "src").exists() else tmp), REPO)
    else:
        url = REPO_URL
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            if token:
                url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        except Exception:
            pass
        if not (REPO / ".git").exists():
            run(["git", "clone", "--quiet", url, str(REPO)])
        run(["git", "fetch", "--quiet", "--tags", "origin"], cwd=REPO, check=False)
        run(["git", "checkout", "--quiet", GIT_REF], cwd=REPO)
        run(["git", "log", "-1", "--format=commit %H (%d) %s"], cwd=REPO)
assert (REPO / "configs" / f"{WORK}.json").exists(), f"Không có configs/{WORK}.json trong repo"

## 4. (Colab) Cài Poppler + HAI môi trường riêng
- `venv-paddle`: paddlevl (transformers ≥ 5, torch ≥ 2.6); `venv-qwen`: Qwen 4-bit + các bước còn lại
  (transformers 4.57.6, giống máy nhóm). Lần đầu ~5-10 phút. Local: bỏ qua (đã cài theo runbook mục 3).

In [ ]:
if not IS_COLAB:
    print("Local: bỏ qua cài đặt.")
else:
    run("apt-get -qq update && apt-get -qq install -y poppler-utils > /dev/null")
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    COMMON_PKGS = ["python-dotenv", "pillow", "numpy", "pdf2image", "opencc", "huggingface_hub", "pytest"]
    if not Path(PADDLE_PY).exists():
        run(["uv", "venv", "-q", "--python", "3.11", "/content/venv-paddle"])
        run(["uv", "pip", "install", "-q", "--python", PADDLE_PY, "torch==2.14.1",
             "--index-url", "https://download.pytorch.org/whl/cu126"])
        run(["uv", "pip", "install", "-q", "--python", PADDLE_PY, "transformers==5.24.0", *COMMON_PKGS])
    if not Path(QWEN_PY).exists():
        run(["uv", "venv", "-q", "--python", "3.11", "/content/venv-qwen"])
        run(["uv", "pip", "install", "-q", "--python", QWEN_PY, "torch==2.5.1", "torchvision==0.20.1",
             "--index-url", "https://download.pytorch.org/whl/cu121"])
        run(["uv", "pip", "install", "-q", "--python", QWEN_PY, "transformers==4.57.6", "accelerate==1.2.1",
             "bitsandbytes==0.50.2", "qwen-vl-utils", *COMMON_PKGS])
for py in (PADDLE_PY, QWEN_PY):
    run([str(py), "-c", "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())"])

## 5. Dữ liệu: PDF nguồn + kiểm tra SHA-256
Colab: `data/<WORK>/03_ocr` là symlink tới Drive, PDF được chép từ Drive. Local: PDF phải nằm đúng chỗ trong
`data/<WORK>/01_sources/` (tên theo `common.source` của config). `raw/*.json` chỉ được thêm run, không bị sửa/xoá.

In [ ]:
cfg = json.loads((REPO / "configs" / f"{WORK}.json").read_text(encoding="utf-8"))
source = REPO / cfg["common"]["source"]
if IS_COLAB:
    work_dir = REPO / "data" / WORK
    work_dir.mkdir(parents=True, exist_ok=True)
    links = {"03_ocr": DRIVE / WORK / "03_ocr"}
    if KEEP_IMAGES_ON_DRIVE:
        links["02_extraction"] = DRIVE / WORK / "02_extraction"
    for name, target in links.items():
        target.mkdir(parents=True, exist_ok=True)
        local = work_dir / name
        if local.is_symlink():
            continue
        if local.exists():                 # repo có sẵn vài file đã commit: chép sang Drive (không ghi đè), rồi link
            for f in local.rglob("*"):
                dst = target / f.relative_to(local)
                if f.is_file() and not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst)
            shutil.rmtree(local)
        local.symlink_to(target, target_is_directory=True)
    logs = REPO / "logs"
    if not logs.is_symlink():
        shutil.rmtree(logs, ignore_errors=True); logs.symlink_to(DRIVE / "logs", target_is_directory=True)
    if not source.exists():
        assert Path(PDF_ON_DRIVE).exists(), f"Không thấy PDF trên Drive: {PDF_ON_DRIVE}"
        source.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PDF_ON_DRIVE, source)
else:
    assert source.exists(), f"Chưa có PDF nguồn: đặt file vào {source} (docs/ocr_assignment.md)"
run([str(QWEN_PY), "-m", "src.ocr.work_config", "--config", f"configs/{WORK}.json", "--verify-source"], cwd=REPO)

## 6. Model: tải nếu chưa có (cần mạng), sau đó chạy offline

In [ ]:
MODELS = ((PADDLE_PY, "paddlevl", "PaddlePaddle/PaddleOCR-VL-1.6"), (QWEN_PY, "qwen", "Qwen/Qwen2.5-VL-3B-Instruct"))
for py, engine, model in MODELS:
    probe = ("from huggingface_hub import try_to_load_from_cache as t; import sys; "
             f"sys.exit(0 if isinstance(t('{model}', 'config.json'), str) else 1)")
    if run([str(py), "-c", probe], check=False) == 0:
        print(f"{model}: đã có trong cache")
    else:
        run([str(py), "-c", f"from huggingface_hub import snapshot_download as s; print(s('{model}'))"],
            env={"HF_HUB_OFFLINE": "0"})
os.environ["HF_HUB_OFFLINE"] = "1"

## 7. (Tuỳ chọn) Giải gói hạt giống — trang đã OCR sẵn sẽ được bỏ qua

In [ ]:
if SEED_ZIP:
    run([str(QWEN_PY), "scripts/unpack_ocr.py", SEED_ZIP, "--data-root", DATA_ROOT], cwd=REPO, check=False)
else:
    print("Không có gói hạt giống.")

## 8. Kiểm tra sẵn sàng + chạy khô (không nạp model): trang nào đã xong, trang nào sẽ OCR, ước lượng thời gian

In [ ]:
run([str(PADDLE_PY), "-m", "src.ocr.preflight", "--engine", "paddlevl"], cwd=REPO)
run([str(QWEN_PY), "-m", "src.ocr.preflight", "--engine", "qwen"], cwd=REPO)
ocr_step("han vie", dry_run=True)

## 9. Trích ảnh trang (chỉ các trang trong `PAGES`; ảnh đã có thì bỏ qua)

In [ ]:
ocr_step("extract")

## 10. Luồng Hán — paddlevl
RTX 3050: ~15-25 s/trang (đo thật). T4 tự dùng float32, chưa đo (ước 1-1,5 lần). Bị ngắt: chạy lại ô này.

In [ ]:
ocr_step("han")

## 11. Luồng Việt — Qwen2.5-VL-3B 4-bit (RTX 3050 ~28-40 s/trang; T4 chưa đo)

In [ ]:
ocr_step("vie")

## 12. Dựng han.xml / vie.xml / TXT và kiểm tra trong `PAGES`
"ĐỦ" = 0 lỗi. Cảnh báo cần người xem (bảng mã ở `docs/ocr_runbook.md` mục 6). 5 trang ngẫu nhiên được in ra để
soát bằng mắt: so ảnh `data/<WORK>/02_extraction/pages/page_NNN.png` với `03_ocr/ocr_raw_text/{han,vie}/NNNN.txt`.

In [ ]:
status = ocr_step("build check", check=False)
print("\nKẾT QUẢ:", "ĐỦ - có thể đóng gói" if status == 0 else "CHƯA ĐỦ - xem lỗi ở trên; thường chỉ cần chạy lại ô 10/11")

## 13. Đóng gói (gửi file .zip cho người điều phối)

In [ ]:
DIST.mkdir(parents=True, exist_ok=True)
run([str(QWEN_PY), "scripts/pack_ocr.py", "--work-id", WORK, "--pages", PAGES, "--member", MEMBER,
     "--data-root", DATA_ROOT, "--out", str(DIST), "--force"], cwd=REPO)
for z in sorted(DIST.glob(f"{WORK}_*_{MEMBER}_*.zip")):
    print(f"{z}  ({z.stat().st_size / 1e6:.1f} MB)")
print("Gửi file .zip ở trên qua Drive/Teams. KHÔNG gửi .env, không commit data/ hay dist/.")

## Ghi chú
- **Mọi người dùng cùng commit/tag.** Mỗi run ghi lại commit, GPU và dtype (`provenance`). T4 dùng float32 cho
  paddlevl nên khoá cache khác RTX 3050 (bf16). Đây không phải lỗi: `--skip-done` bỏ qua theo số trang, và khi
  ghép, `build_layers` chọn run theo manifest, rồi run hợp lệ, rồi run mới nhất.
- Xử lý lỗi: `docs/ocr_runbook.md` mục 5. OOM thì script tự thử lại với `*_MAX_PIXELS_TOKENS=768`. Bị ngắt thì *Run all* lại.
- Chạy tay một trang (vd OCR lại trang 208 với dải thấp hơn):
  `QWEN_BAND_INCHES=1.0 <QWEN_PY> -m src.ocr.ocr --config configs/CPO_18.json --layer vie --engines qwen --pages 208`